In [13]:
%reset -f

In [5]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer

In [6]:
df = pd.read_json('../input/dataset.json', lines=True)

In [7]:
df

,reviewText,overall
0,"If you've read other max brand westerns, you k...",5
1,"Love Max, always a fun twist",5
2,"As usual for him, a good book",5
3,great book,5
4,ALMOST BEEN TOO LONG SINCE I READ IT. GOOD REA...,4
...,...,...
879753,(7 year old's review)\n\nThey took a long time...,5
879754,So much fun! Very cute and well done. Pirates ...,5
879755,If you haven't noticed I love Kia Meches writi...,5
879756,"I liked Secret. Sky,King,Parrish. I didn't lik...",5


In [8]:
corpus = []
for review in df['reviewText']:
    res = re.sub('[^a-zA-Z]', ' ', review) #Replace every non (^) letter(a-zA-Z) character by a space
    res = res.lower()
    res = res.split()
    lemmatizer = WordNetLemmatizer()
    all_stopwords = stopwords.words('english')
    all_stopwords.remove("not")
    res = [lemmatizer.lemmatize(word) for word in res if not word in set(all_stopwords)] # Apply stemming to every words except the stopwords from the english language
    res = ' '.join(res)
    corpus.append(res)

In [9]:
corpus

['read max brand western know expect not treat let strap six gun gotta ride',
 'love max always fun twist',
 'usual good book',
 'great book',
 'almost long since read good read would not finished',
 'enjoyed book read writer read book read good story line',
 'good read western fan great price',
 'hoping find one book form story look like good get far series',
 'typical l amour western not one best enjoyable nevertheless l amour always good read good excapeism',
 'prejudiced toward anything louis lamour writes easily one best writer th century',
 'year reading western none better louis l amour book like rest correct place include foul language sex',
 'tragic love story leave series question love love much love not exactly reciprocal',
 'true story amazing thing got away long nd good person',
 'fun adventure read',
 'amazing tale man got away not believe managed get away thing',
 'great book read two flight could not stop reading even better movie highly recommend adding collection',
 '

# TF-IDF Method

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [11]:
tfidf = TfidfVectorizer(max_features=1000)
X = tfidf.fit_transform(corpus).toarray()
y = df["overall"].values

In [12]:
from sklearn.model_selection import train_test_split

X_tfidf_train, X_tfidf_test, y_tfidf_train, y_tfidf_test = train_test_split(X, y, test_size=0.33, random_state=50)

In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

from sklearn.metrics import accuracy_score,make_scorer, classification_report
from sklearn.model_selection import GridSearchCV


In [14]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_tfidf_train_encoded = label_encoder.fit_transform(y_tfidf_train)

In [15]:
model = Sequential()
model.add(Dense(128, input_shape=(X_tfidf_train.shape[1],), activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(5, activation='softmax'))  # 5 classes for ratings 1 to 5
optimizer = Adam(learning_rate=0.001)
loss = SparseCategoricalCrossentropy()
model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

# Training the model
model.fit(X_tfidf_train, y_tfidf_train_encoded, epochs=5, batch_size=32, validation_split=0.1)

# Evaluating the model
y_pred_proba = model.predict(X_tfidf_test)
y_pred = label_encoder.inverse_transform(y_pred_proba.argmax(axis=1))

# Evaluation
accuracy = accuracy_score(y_tfidf_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

# Additional metrics
print(classification_report(y_tfidf_test, y_pred))



Epoch 1/5


16578/16578 [==============================] - 60s 4ms/step - loss: 0.7815 - accuracy: 0.6800 - val_loss: 0.7682 - val_accuracy: 0.6838
Epoch 2/5
16578/16578 [==============================] - 57s 3ms/step - loss: 0.7564 - accuracy: 0.6894 - val_loss: 0.7588 - val_accuracy: 0.6860
Epoch 3/5
16578/16578 [==============================] - 57s 3ms/step - loss: 0.7431 - accuracy: 0.6945 - val_loss: 0.7568 - val_accuracy: 0.6874
Epoch 4/5
16578/16578 [==============================] - 57s 3ms/step - loss: 0.7325 - accuracy: 0.6991 - val_loss: 0.7566 - val_accuracy: 0.6879
Epoch 5/5
9073/9073 [==============================] - 17s 2ms/step
Accuracy: 0.69
              precision    recall  f1-score   support

           1       0.51      0.38      0.44      7035
           2       0.38      0.11      0.17      7952
           3       0.46      0.37      0.41     24827
           4       0.49      0.22      0.30     65927
           5       0.74      0.94      0.83    184580

    

# Single prediction 

In [16]:
def predict_rating(text):
    corpus = []
    res = re.sub('[^a-zA-Z]', ' ', text.lower())
    res = res.split()
    lemmatizer = WordNetLemmatizer()
    all_stopwords = stopwords.words('english')
    all_stopwords.remove("not")
    res = [lemmatizer.lemmatize(word) for word in res if not word in set(all_stopwords)]
    res = ' '.join(res)
    corpus.append(res)
    X = tfidf.transform(corpus).toarray()  # Use transform, not fit_transform
    prediction_proba = model.predict(X)
    predicted_class = label_encoder.inverse_transform([prediction_proba.argmax()])[0]
    print(f'Predicted Rating: {predicted_class}')

In [22]:
text = "i have seen better."
predict_rating(text)

1/1 [==============================] - 0s 23ms/step
Predicted Rating: 5


As you can see, some sentences are not well guess. It might be because of :
- Lack of datas on low rating 
- Use of TF-IDF that doesn't take the context of words into account

So the next steps should be :

- Getting more data of bad rated reviews (1 to 3)
- Learn about Contextual Word Embedding